# VLM-DENTAL — Trace Generation Workspace

This notebook handles autonomous CoT trace generation using LangGraph and vLLM.

**Architecture**: Generation and verification are **decoupled**.
- **Cell 7a** generates raw traces at full speed (no rate limit when using local vLLM).
- **Cell 7b** verifies pending traces via external APIs (rate-limited by ProviderPool).
- **Cell 8** auto-pushes verified traces to GitHub after each session.
- Both generate and verify can run simultaneously and support resume.


## 1. Environment Setup & Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**(Optional) Fresh Start Cleanup:**
Run this cell ONLY if you need to completely delete the VLM-DENTAL folder from your Google Drive to start over.

In [ ]:
# Uncomment the line below to delete the folder, then run the cell
# !rm -rf /content/drive/MyDrive/VLM-DENTAL

In [ ]:
import os

# Set this to True to save the 10GB dataset and repo itself to Google Drive.
# Set this to False to keep the repo/dataset in temporary Colab storage.
SAVE_DATASET_AND_CODE_TO_DRIVE = False

# Set this to True to save generated YOLO model outputs to Google Drive.
# Set this to False to keep YOLO weights/results in the active repo clone under /content/.../data/models.
SAVE_YOLO_RESULTS_TO_DRIVE = True

drive_path = "/content/drive/MyDrive/VLM-DENTAL"
colab_path = "/content/VLM-DENTAL"
work_dir = drive_path if SAVE_DATASET_AND_CODE_TO_DRIVE else colab_path
models_root = f"{drive_path}/data/models" if SAVE_YOLO_RESULTS_TO_DRIVE else f"{work_dir}/data/models"
os.environ["YOLO_MODELS_ROOT"] = models_root

In [ ]:
import os

if SAVE_DATASET_AND_CODE_TO_DRIVE:
    os.chdir("/content/drive/MyDrive")
else:
    os.chdir("/content")

if not os.path.exists("VLM-DENTAL"):
    os.system("git clone https://github.com/rezaxr14/VLM-DENTAL.git")

os.chdir(work_dir)
os.system("git pull")

# Ensure output directories exist
os.makedirs(models_root, exist_ok=True)
if SAVE_YOLO_RESULTS_TO_DRIVE:
    os.makedirs(f"{drive_path}/data/traces", exist_ok=True)

# --- One-time: Rename legacy trace file to .old ---
legacy_path = "data/traces/train_cot_traces.jsonl"
backup_path = "data/traces/train_cot_traces.jsonl.old"
if os.path.exists(legacy_path) and not os.path.exists(backup_path):
    # Check if this is the old legacy file (not our new verified output)
    file_size = os.path.getsize(legacy_path)
    if file_size > 0:
        os.rename(legacy_path, backup_path)
        print(f"Renamed legacy traces ({file_size/1024:.0f}KB) to {backup_path}")
    else:
        print("Legacy trace file is empty, no rename needed.")
elif os.path.exists(backup_path):
    print(f"Legacy backup already exists at {backup_path}")

## 2. Install Dependencies

In [ ]:
# Install the project and all its requirements
!pip install -e .
!pip install python-dotenv pandas pillow google-genai huggingface_hub ultralytics
!pip install vllm langgraph

## 3. Configure Credentials
Loads credentials in two layers:
1. **`.env` file** (baseline) — upload your `.env` to the project root. All keys from it are loaded first.
2. **Colab Secrets tab** (override) — any key set here takes priority over `.env`.

**Required keys:** `GITHUB_TOKEN` + at least one verifier API key.

In [ ]:
import os
import shutil

# --- Layer 1: Load from .env file (baseline) ---
# If you uploaded a .env to the project root, all keys are loaded here.
# If no .env exists, copy from .env.example as a starting template.
if not os.path.exists('.env') and os.path.exists('.env.example'):
    shutil.copy('.env.example', '.env')
    print('Created .env from .env.example (fill in your keys or upload your own .env)')

from dotenv import load_dotenv
dotenv_loaded = load_dotenv('.env', override=False)
if dotenv_loaded:
    print('Loaded credentials from .env file')
else:
    print('No .env file found — relying on Colab Secrets tab only')

# --- Layer 2: Colab Secrets tab (overrides .env) ---
try:
    from google.colab import userdata

    secret_keys = [
        # API keys
        'GEMINI_API_KEY', 'NVIDIA_API_KEY', 'GROQ_API_KEY', 'OPENROUTER_API_KEY',
        # Verifier models
        'NVIDIA_VERIFIER_MODEL', 'GROQ_VERIFIER_MODEL',
        'OPENROUTER_VERIFIER_MODEL', 'GEMINI_VERIFIER_MODEL',
        # Generator config
        'GENERATOR_PROVIDER', 'GENERATOR_MODEL',
        'GENERATOR_COOLDOWN_SECONDS', 'GENERATOR_RPD_LIMIT',
        'NVIDIA_GENERATOR_MODEL', 'GROQ_GENERATOR_MODEL',
        'OPENROUTER_GENERATOR_MODEL', 'GEMINI_GENERATOR_MODEL',
        # Verifier rate limits
        'API_COOLDOWN_SECONDS', 'API_RPD_LIMIT',
        # HuggingFace
        'HF_TOKEN',
        # GitHub (for auto-pushing verified traces)
        'GITHUB_TOKEN',
    ]
    overridden = []
    for key in secret_keys:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val  # overrides .env value
                overridden.append(key)
        except Exception:
            pass

    if overridden:
        print(f'Colab Secrets overrode {len(overridden)} key(s): {overridden}')
    else:
        print('No Colab Secrets found — using .env values only')
except ImportError:
    print('Colab Secrets API not available — using .env values only')

# Ensure local vLLM defaults are set if neither source provided them
os.environ.setdefault('GENERATOR_PROVIDER', 'local')
os.environ.setdefault('GENERATOR_MODEL', 'Qwen/Qwen3.5-9B')
os.environ.setdefault('LOCAL_VLLM_BASE_URL', 'http://localhost:8000/v1')

# Summary
print()
print('--- Active Configuration ---')
print(f"GENERATOR_PROVIDER : {os.environ.get('GENERATOR_PROVIDER', 'NOT SET')}")
print(f"GENERATOR_MODEL    : {os.environ.get('GENERATOR_MODEL', 'NOT SET')}")
print(f"GITHUB_TOKEN       : {'SET' if os.environ.get('GITHUB_TOKEN', '').strip() and not os.environ.get('GITHUB_TOKEN', '').startswith('your_') else 'NOT SET'}")
active_verifiers = [p for p in ['NVIDIA', 'GROQ', 'OPENROUTER', 'GEMINI']
                    if os.environ.get(f'{p}_API_KEY', '').strip()
                    and not os.environ.get(f'{p}_API_KEY', '').startswith('your_')]
print(f"Active verifiers   : {active_verifiers if active_verifiers else 'NONE (add API keys!)'}")

## 4. Model Cache Setup
Sets `HF_HOME` to a folder inside the project so the Qwen3.5-9B model weights are cached and reused across cell re-runs.

In [ ]:
import os

# Cache HuggingFace models inside the project to avoid re-downloading
vllm_cache_dir = os.path.join(os.getcwd(), 'data', 'models', 'vllm_cache')
os.makedirs(vllm_cache_dir, exist_ok=True)

os.environ['HF_HOME'] = vllm_cache_dir
os.environ['HF_HUB_CACHE'] = os.path.join(vllm_cache_dir, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(vllm_cache_dir, 'hub')

model_name = os.environ.get('GENERATOR_MODEL', 'Qwen/Qwen3.5-9B')

# Check if model is already cached
hub_dir = os.path.join(vllm_cache_dir, 'hub')
model_slug = model_name.replace('/', '--')
cached_path = os.path.join(hub_dir, f'models--{model_slug}')

if os.path.exists(cached_path):
    cached_size_mb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, dn, fns in os.walk(cached_path) for f in fns
    ) / (1024 * 1024)
    print(f'Model already cached at {cached_path} ({cached_size_mb:.0f} MB)')
    print('vLLM will reuse these weights without re-downloading.')
else:
    print(f'Model not yet cached. vLLM will download {model_name} to {vllm_cache_dir}')
    print('This download happens once and is reused on subsequent runs.')

## 5. Dataset Download & Cleanup
Run this to download the dataset if you haven't already. It will extract and structure it automatically.

In [ ]:
!python download_and_cleanup.py

In [ ]:
# Delete unused partial datasets to save space
!rm -rf data/dentex/DENTEX/training_data/disease
!rm -rf data/dentex/DENTEX/training_data/quadrant
!rm -rf data/dentex/DENTEX/training_data/unlabelled/

!rm -rf data/dentex/DENTEX/testing_data/disease
!rm -rf data/dentex/DENTEX/testing_data/quadrant

## 6. Install & Stand up vLLM Server
Starts the OpenAI-compatible API server in the background for Qwen3.5-9B.

**Note**: This cell is only needed when `GENERATOR_PROVIDER=local`. If using an external API for generation, skip to Cell 7a.

In [ ]:
import subprocess
import time
import os
import urllib.request

model_name = os.environ.get('GENERATOR_MODEL', 'Qwen/Qwen3.5-9B')
vllm_cache_dir = os.environ.get('HF_HOME', 'data/models/vllm_cache')
port = '8000'

print(f'Starting vLLM server for {model_name}...')
print(f'Model cache: {vllm_cache_dir}')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

vllm_process = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', model_name,
    '--port', port,
    '--max-model-len', '4096',
    '--download-dir', os.path.join(vllm_cache_dir, 'hub'),
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

# Health-check polling loop (ping /v1/models every 5s, timeout 5 min)
base_url = f'http://localhost:{port}/v1'
max_wait = 300  # 5 minutes
poll_interval = 5
elapsed = 0

print('Waiting for vLLM server to become ready', end='')
while elapsed < max_wait:
    try:
        req = urllib.request.Request(f'{base_url}/models')
        with urllib.request.urlopen(req, timeout=3) as resp:
            if resp.getcode() == 200:
                print(f'\nvLLM server is READY! (took {elapsed}s)')
                break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(poll_interval)
    elapsed += poll_interval
else:
    print(f'\nERROR: vLLM server did not become ready within {max_wait}s.')
    print('Check GPU memory and model compatibility.')
    raise RuntimeError('vLLM server startup timeout')

## 7a. Generate Traces (GPU-bound)
Runs the LangGraph loop for each image, writing raw (unverified) traces to `train_cot_traces_unverified.jsonl`.

- When `GENERATOR_PROVIDER=local`: no rate limit, runs as fast as the GPU allows.
- When using an external API: rate-limited by `GeneratorPool`.
- **Resumable**: skips images already in the unverified file.

This cell can run simultaneously with Cell 7b (verify).

In [ ]:
!python scripts/run_trace_gen.py --mode generate --split train

## 7b. Verify Traces (API-bound)
Reads unverified traces, verifies each via the ProviderPool (external API round-robin with rate limits), and promotes passing traces to `train_cot_traces.jsonl`.

- Rate-limited by `ProviderPool` (5-min cooldown, 10 RPD per provider).
- **Resumable**: tracks already-verified image IDs.
- Can be re-run any time to verify more pending traces.

This cell can run simultaneously with Cell 7a (generate).

In [ ]:
!python scripts/run_trace_gen.py --mode verify --split train

## 8. Push Verified Traces to GitHub
Automatically commits and pushes new verified traces to the repo using the `GITHUB_TOKEN` from Colab Secrets or `.env`.
This runs after each verification session so your traces are always synced.

In [ ]:
import os
import subprocess
import json

verified_path = 'data/traces/train_cot_traces.jsonl'
unverified_path = 'data/traces/train_cot_traces_unverified.jsonl'

# Check if there are traces to push
if not os.path.exists(verified_path) or os.path.getsize(verified_path) == 0:
    print('No verified traces to push yet. Run Cell 7b first.')
else:
    # Configure git identity
    subprocess.run(['git', 'config', 'user.email', 'rezaxr14@gmail.com'])
    subprocess.run(['git', 'config', 'user.name', 'Reza Nadimi'])

    # Set up authentication via GITHUB_TOKEN
    github_token = os.environ.get('GITHUB_TOKEN', '')
    if not github_token or github_token.startswith('your_'):
        print('WARNING: GITHUB_TOKEN not set. Cannot push.')
        print('Set it in the Colab Secrets tab or in your .env file.')
    else:
        # Get current remote URL and inject token
        result = subprocess.run(['git', 'remote', 'get-url', 'origin'],
                                capture_output=True, text=True)
        remote_url = result.stdout.strip()

        # Convert https://github.com/user/repo.git -> https://<token>@github.com/user/repo.git
        if 'github.com' in remote_url and '@' not in remote_url:
            auth_url = remote_url.replace(
                'https://github.com',
                f'https://{github_token}@github.com'
            )
            subprocess.run(['git', 'remote', 'set-url', 'origin', auth_url])

        # Stage trace files
        subprocess.run(['git', 'add', verified_path])
        if os.path.exists(unverified_path):
            subprocess.run(['git', 'add', unverified_path])

        # Count verified traces for commit message
        n_traces = 0
        with open(verified_path, 'r') as f:
            n_traces = sum(1 for line in f if line.strip())

        # Check if there are changes to commit
        status = subprocess.run(['git', 'diff', '--cached', '--name-only'],
                                capture_output=True, text=True)
        if status.stdout.strip():
            commit_msg = f'data: {n_traces} verified CoT traces (auto-push from Colab)'
            subprocess.run(['git', 'commit', '-m', commit_msg])
            push_result = subprocess.run(['git', 'push'],
                                         capture_output=True, text=True)
            if push_result.returncode == 0:
                print(f'Successfully pushed {n_traces} verified traces to GitHub!')
            else:
                print(f'Push failed: {push_result.stderr}')
                print('You may need to pull first: !git pull --rebase')
        else:
            print('No new changes to push (traces already up-to-date on GitHub).')

## 9. Status Dashboard
Shows generation progress, verification progress, and pool capacity at a glance.

In [ ]:
import json
import os

unverified_path = 'data/traces/train_cot_traces_unverified.jsonl'
verified_path = 'data/traces/train_cot_traces.jsonl'
legacy_path = 'data/traces/train_cot_traces.jsonl.old'

def count_ids(path):
    ids = set()
    if os.path.exists(path):
        with open(path, 'r') as f:
            for line in f:
                try:
                    r = json.loads(line.strip())
                    if 'image_id' in r:
                        ids.add(int(r['image_id']))
                except Exception:
                    pass
    return ids

unverified_ids = count_ids(unverified_path)
verified_ids = count_ids(verified_path)
pending = unverified_ids - verified_ids

print('=' * 50)
print('TRACE GENERATION STATUS DASHBOARD')
print('=' * 50)
print(f'Generated (unverified) : {len(unverified_ids)}')
print(f'Verified (SFT-ready)   : {len(verified_ids)}')
print(f'Pending verification   : {len(pending)}')

if os.path.exists(legacy_path):
    legacy_size = os.path.getsize(legacy_path) / 1024
    print(f'Legacy backup (.old)   : {legacy_size:.0f} KB')

print('=' * 50)

# Also show pool status
print()
!python scripts/run_trace_gen.py --mode verify --status-only

## 10. Download Generated Traces Locally

In [ ]:
from google.colab import files
import os

# Download the verified (SFT-ready) traces
verified_path = 'data/traces/train_cot_traces.jsonl'
if os.path.exists(verified_path) and os.path.getsize(verified_path) > 0:
    files.download(verified_path)
    print(f'Downloaded: {verified_path}')
else:
    print('No verified traces yet. Run Cell 7b to verify pending traces.')

# Optionally also download unverified traces
unverified_path = 'data/traces/train_cot_traces_unverified.jsonl'
if os.path.exists(unverified_path) and os.path.getsize(unverified_path) > 0:
    files.download(unverified_path)
    print(f'Downloaded: {unverified_path}')